In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_1.py ──
"""
Shared infrastructure for MLFP03 Exercise 1 — Feature Engineering
and Feature Selection on ICU data.

Contains:
    - ICU multi-table loading with temporal casts
    - Point-in-time feature builders (vitals, medications, labs)
    - ExperimentTracker / ConnectionManager setup
    - Shared prep helpers for feature-selection methods
    - Plotting helpers for feature rankings

Technique-specific code (mutual_info, RFE, Lasso paths, schema
validation) does NOT belong here — it lives in the per-technique files.
"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from dotenv import load_dotenv

from kailash.db import ConnectionManager
from kailash_ml import DataExplorer
from kailash_ml.engines.experiment_tracker import ExperimentTracker



# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()
load_dotenv()

OUTPUT_DIR = Path("outputs") / "mlfp03_ex1_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_DB = "sqlite:///mlfp03_experiments.db"
EXPERIMENT_NAME = "mlfp03_healthcare_features"

_DT_FMT = "%Y-%m-%d %H:%M:%S"

VITAL_COLS: list[str] = [
    "heart_rate",
    "systolic_bp",
    "diastolic_bp",
    "temperature",
    "spo2",
    "respiratory_rate",
]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — ICU multi-table
# ════════════════════════════════════════════════════════════════════════


def load_icu_tables() -> dict[str, pl.DataFrame]:
    """Load all five ICU tables and cast timestamp columns to datetime.

    Returns a dict with keys: patients, admissions, vitals (long format),
    medications, labs.

    Vitals are returned in LONG format (columns: admission_id, patient_id,
    timestamp, vital_name, value) — the monolithic exercise unpivots them
    inline, but every technique file wants the long form.
    """
    loader = MLFPDataLoader()
    patients = loader.load("mlfp02", "icu_patients.parquet")
    admissions = loader.load("mlfp02", "icu_admissions.parquet")
    vitals = loader.load("mlfp02", "icu_vitals.parquet")
    medications = loader.load("mlfp02", "icu_medications.parquet")
    labs = loader.load("mlfp02", "icu_labs.parquet")

    # Cast timestamps — polars reads them as strings from the parquet
    admissions = admissions.with_columns(
        pl.col("admit_time").str.to_datetime(_DT_FMT),
        pl.col("discharge_time").str.to_datetime(_DT_FMT),
    )
    medications = medications.with_columns(
        pl.col("start_time").str.to_datetime(_DT_FMT),
        pl.col("end_time").str.to_datetime(_DT_FMT),
    )
    if "timestamp" in labs.columns and labs["timestamp"].dtype == pl.String:
        labs = labs.with_columns(pl.col("timestamp").str.to_datetime(_DT_FMT))

    # Vitals: attach patient_id via admissions join, cast timestamp, unpivot
    vitals = vitals.join(
        admissions.select(["admission_id", "patient_id"]),
        on="admission_id",
        how="left",
    ).with_columns(pl.col("timestamp").str.to_datetime(_DT_FMT))

    present = [c for c in VITAL_COLS if c in vitals.columns]
    if present:
        vitals = vitals.unpivot(
            present,
            index=["admission_id", "patient_id", "timestamp"],
            variable_name="vital_name",
            value_name="value",
        )

    return {
        "patients": patients,
        "admissions": admissions,
        "vitals": vitals,
        "medications": medications,
        "labs": labs,
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE BUILDERS — point-in-time aggregates
# ════════════════════════════════════════════════════════════════════════


def build_vital_features(
    vitals: pl.DataFrame, admissions: pl.DataFrame
) -> pl.DataFrame:
    """Aggregate long-format vitals per admission with temporal correctness.

    Only uses vital readings recorded BETWEEN admit_time and discharge_time
    for each admission. Returns one row per admission with columns:
        {vital}_{mean,std,min,max,range,trend,count,cv}
    """
    # Vitals already carries admission_id and patient_id. Join only to pull
    # in the admit/discharge window; drop admissions' patient_id on the
    # way in to avoid duplicate columns.
    filtered = vitals.join(
        admissions.select("admission_id", "admit_time", "discharge_time"),
        on="admission_id",
        how="inner",
    ).filter(
        (pl.col("timestamp") >= pl.col("admit_time"))
        & (pl.col("timestamp") <= pl.col("discharge_time"))
    )

    names = filtered["vital_name"].unique().to_list()
    aggs: list[pl.DataFrame] = []
    for vital in names:
        agg = (
            filtered.filter(pl.col("vital_name") == vital)
            .group_by("admission_id")
            .agg(
                pl.col("value").mean().alias(f"{vital}_mean"),
                pl.col("value").std().alias(f"{vital}_std"),
                pl.col("value").min().alias(f"{vital}_min"),
                pl.col("value").max().alias(f"{vital}_max"),
                (pl.col("value").max() - pl.col("value").min()).alias(f"{vital}_range"),
                (pl.col("value").last() - pl.col("value").first()).alias(
                    f"{vital}_trend"
                ),
                pl.col("value").count().alias(f"{vital}_count"),
                (pl.col("value").std() / pl.col("value").mean()).alias(f"{vital}_cv"),
            )
        )
        aggs.append(agg)

    # Merge vital aggregates via full-outer join with coalesced key.
    out = aggs[0]
    for a in aggs[1:]:
        out = out.join(a, on="admission_id", how="full", coalesce=True)
    return out


def build_medication_features(
    medications: pl.DataFrame, admissions: pl.DataFrame
) -> pl.DataFrame:
    """Flag high-risk medications and count distinct drugs per admission."""
    return (
        medications.join(
            admissions.select(
                "patient_id", "admission_id", "admit_time", "discharge_time"
            ),
            on="admission_id",
            how="inner",
        )
        .filter(
            (pl.col("start_time") >= pl.col("admit_time"))
            & (pl.col("start_time") <= pl.col("discharge_time"))
        )
        .group_by("admission_id")
        .agg(
            pl.col("drug_name").n_unique().alias("n_unique_medications"),
            pl.col("drug_name").count().alias("n_medication_doses"),
            pl.col("drug_name")
            .str.contains("(?i)vasopressor|norepinephrine|dopamine")
            .any()
            .alias("received_vasopressors"),
            pl.col("drug_name")
            .str.contains("(?i)antibiotic|vancomycin|meropenem")
            .any()
            .alias("received_antibiotics"),
            pl.col("drug_name")
            .str.contains("(?i)propofol|midazolam|fentanyl")
            .any()
            .alias("received_sedation"),
        )
    )


def build_lab_features(labs: pl.DataFrame, admissions: pl.DataFrame) -> pl.DataFrame:
    """Aggregate lab results per admission with abnormal-flag counts."""
    return (
        labs.join(
            admissions.select("admission_id", "admit_time", "discharge_time"),
            on="admission_id",
            how="inner",
        )
        .filter(
            (pl.col("timestamp") >= pl.col("admit_time"))
            & (pl.col("timestamp") <= pl.col("discharge_time"))
        )
        .group_by("admission_id")
        .agg(
            pl.col("test_name").n_unique().alias("n_unique_labs"),
            pl.col("value").count().alias("n_lab_results"),
            (pl.col("flag") != "normal").sum().alias("n_abnormal_labs"),
            pl.col("value").mean().alias("lab_value_mean"),
            pl.col("value").std().alias("lab_value_std"),
        )
    )


def build_full_feature_frame(tables: dict[str, pl.DataFrame]) -> pl.DataFrame:
    """End-to-end feature matrix used by every technique file.

    Composes patients + admissions with vital / medication / lab / derived /
    interaction features. Every technique file calls this so the starting
    feature matrix is identical — only the SELECTION method differs.
    """
    patients = tables["patients"]
    admissions = tables["admissions"]

    patient_admissions = patients.join(admissions, on="patient_id", how="inner")

    features = patient_admissions.clone()
    vf = build_vital_features(tables["vitals"], admissions)
    features = features.join(vf, on="admission_id", how="left")

    # Vital stat columns (std, trend, cv, ...) are legitimately null
    # when a patient has only 1 reading for a given vital, or when a
    # vital was never sampled. Fill with 0 so downstream selection
    # methods (sklearn) see no nulls.
    vital_stat_cols = [
        c
        for c in features.columns
        if any(
            c.endswith(f"_{s}")
            for s in ("mean", "std", "min", "max", "range", "trend", "count", "cv")
        )
    ]
    features = features.with_columns(
        *[pl.col(c).fill_null(0.0) for c in vital_stat_cols]
    )

    mf = build_medication_features(tables["medications"], admissions)
    features = features.join(mf, on="admission_id", how="left")

    lf = build_lab_features(tables["labs"], admissions)
    features = features.join(lf, on="admission_id", how="left")

    # Derived features
    features = features.with_columns(
        (pl.col("n_abnormal_labs") / pl.col("n_lab_results").clip(lower_bound=1)).alias(
            "abnormal_lab_ratio"
        ),
        (pl.col("n_medication_doses") / pl.col("los_days").clip(lower_bound=1)).alias(
            "medication_intensity"
        ),
        (pl.col("n_lab_results") / pl.col("los_days").clip(lower_bound=1)).alias(
            "lab_intensity"
        ),
        (pl.col("n_unique_medications") > 10).alias("polypharmacy_flag"),
    )

    # Null fills for patients without meds / labs
    fill_int = [
        "n_unique_medications",
        "n_medication_doses",
        "n_unique_labs",
        "n_lab_results",
        "n_abnormal_labs",
    ]
    fill_bool = [
        "received_vasopressors",
        "received_antibiotics",
        "received_sedation",
        "polypharmacy_flag",
    ]
    fill_float = [
        "abnormal_lab_ratio",
        "medication_intensity",
        "lab_intensity",
        "lab_value_mean",
        "lab_value_std",
    ]
    features = features.with_columns(
        *[pl.col(c).fill_null(0) for c in fill_int if c in features.columns],
        *[pl.col(c).fill_null(False) for c in fill_bool if c in features.columns],
        *[pl.col(c).fill_null(0.0) for c in fill_float if c in features.columns],
    )

    # Interactions (clinical domain knowledge)
    cols = features.columns
    exprs: list[pl.Expr] = []
    if "heart_rate_mean" in cols and "systolic_bp_mean" in cols:
        exprs.append(
            (
                pl.col("heart_rate_mean")
                / pl.col("systolic_bp_mean").clip(lower_bound=1)
            ).alias("shock_index")
        )
    if "systolic_bp_mean" in cols and "diastolic_bp_mean" in cols:
        exprs.append(
            ((pl.col("systolic_bp_mean") + 2 * pl.col("diastolic_bp_mean")) / 3).alias(
                "map_mean"
            )
        )
    if "temperature_mean" in cols and "heart_rate_mean" in cols:
        exprs.append(
            (pl.col("temperature_mean") * pl.col("heart_rate_mean")).alias(
                "fever_tachycardia"
            )
        )
    exprs.append(
        (pl.col("medication_intensity") * pl.col("abnormal_lab_ratio")).alias(
            "treatment_burden_score"
        )
    )
    features = features.with_columns(*exprs)

    # Fill any nulls introduced by the interactions
    for name in (
        "shock_index",
        "map_mean",
        "fever_tachycardia",
        "treatment_burden_score",
    ):
        if name in features.columns:
            features = features.with_columns(pl.col(name).fill_null(0.0))

    return features


# ════════════════════════════════════════════════════════════════════════
# SELECTION INPUT PREP
# ════════════════════════════════════════════════════════════════════════


def prepare_selection_inputs(
    features: pl.DataFrame,
) -> tuple[list[str], np.ndarray, np.ndarray]:
    """Return (feature_cols, X, y_binary) for every feature-selection method.

    - Drops ID columns and the target from the feature matrix
    - Coerces bool/int/float columns only (selection methods require numeric)
    - Replaces NaN / inf with bounded numbers
    - Builds a binary target: mortality if present, otherwise los_days > median
    """
    id_cols = {"patient_id", "admission_id", "admit_time", "discharge_time"}
    target_col = "mortality" if "mortality" in features.columns else "los_days"
    exclude = id_cols | {target_col}

    numeric_dtypes = {pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Boolean}
    feature_cols = [
        c
        for c in features.columns
        if c not in exclude and features[c].dtype in numeric_dtypes
    ]

    X = features.select(feature_cols).to_numpy().astype(np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)

    if target_col == "mortality":
        y = features["mortality"].to_numpy().astype(np.float64).ravel()
    else:
        median_los = features["los_days"].median()
        y = (
            (features["los_days"] > median_los)
            .cast(pl.Int32)
            .to_numpy()
            .ravel()
            .astype(np.float64)
        )
    y_binary = (
        (y > np.median(y)).astype(int) if target_col != "mortality" else y.astype(int)
    )
    return feature_cols, X, y_binary


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING
# ════════════════════════════════════════════════════════════════════════


async def setup_tracking() -> tuple[ConnectionManager, ExperimentTracker, str]:
    """Initialize ConnectionManager + ExperimentTracker and create the M3
    experiment. Every technique file in ex_1 logs into the same experiment
    so selection runs are directly comparable.
    """
    conn = ConnectionManager(EXPERIMENT_DB)
    await conn.initialize()
    tracker = ExperimentTracker(conn)
    experiment_id = await tracker.create_experiment(
        name=EXPERIMENT_NAME,
        description="Feature engineering experiments on ICU data — Module 3",
        tags=["mlfp03", "healthcare", "feature-engineering"],
    )
    return conn, tracker, experiment_id


def setup_tracking_sync() -> tuple[ConnectionManager, ExperimentTracker, str]:
    """Sync wrapper for setup_tracking — convenience for non-async files."""
    return asyncio.run(setup_tracking())


async def log_selection_run(
    tracker: ExperimentTracker,
    experiment_id: str,
    *,
    run_name: str,
    method: str,
    selected_features: list[str],
    total_features: int,
    extra_params: dict[str, str] | None = None,
    extra_metrics: dict[str, float] | None = None,
) -> str:
    """Log a feature-selection run to ExperimentTracker. Returns run id."""
    params = {
        "method": method,
        "n_features_total": str(total_features),
        "n_features_selected": str(len(selected_features)),
        "selected_features": ",".join(selected_features[:30]),
    }
    if extra_params:
        params.update(extra_params)
    metrics = {
        "n_features_selected": float(len(selected_features)),
        "selection_ratio": float(len(selected_features)) / max(1, total_features),
    }
    if extra_metrics:
        metrics.update(extra_metrics)

    async with tracker.run(experiment_id, run_name=run_name) as run:
        await run.log_params(params)
        await run.log_metrics(metrics)
        await run.set_tag("domain", "clinical")
        await run.set_tag("selection_family", method)
        run_id = getattr(run, "id", run_name)
    return run_id


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_ranking(
    title: str, ranking: list[tuple[str, float]], *, top: int = 15
) -> None:
    """Print a ranked feature list with a simple ASCII bar chart."""
    print(f"\n=== {title} ===")
    print(f"{'Feature':<35} {'Score':>10}")
    print("-" * 48)
    if not ranking:
        print("  (empty ranking)")
        return
    max_score = max(abs(s) for _, s in ranking[:top]) or 1.0
    for name, score in ranking[:top]:
        bar_len = int(abs(score) / max_score * 20)
        bar = "#" * bar_len
        print(f"  {name:<33} {score:>10.4f}  {bar}")


def save_ranking_csv(
    ranking: list[tuple[str, float]], filename: str, score_col: str = "score"
) -> Path:
    """Persist a ranking as CSV into OUTPUT_DIR. Returns the file path."""
    path = OUTPUT_DIR / filename
    pl.DataFrame(
        {"feature": [n for n, _ in ranking], score_col: [s for _, s in ranking]}
    ).write_csv(path)
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 1.4: Embedded Feature Selection (L1 / Lasso Path)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Use L1 regularisation to drive coefficients to exactly zero
#   - Walk a regularisation path (multiple C values)
#   - Apply to DBS Bank card-fraud scoring where latency dominates
#
# PREREQUISITES: 03_wrapper_selection.py
# ESTIMATED TIME: ~25 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler



## THEORY — L1 Drives Coefficients To Zero


In [ ]:
# L1 (Lasso) adds a penalty proportional to sum(|w_j|). The corner at
# zero means weak coefficients are pushed all the way to zero, not
# merely close to zero. Every zero coefficient is an eliminated
# feature — selection happens INSIDE the training loop.



## TASK 2 — BUILD: scale inputs and define the C grid


In [ ]:
tables = load_icu_tables()
features = build_full_feature_frame(tables)
feature_cols, X_sel, y_binary = prepare_selection_inputs(features)

print("\n" + "=" * 70)
print("  Embedded Selection — L1 Logistic Regression")
print("=" * 70)
print(f"  Features: {len(feature_cols)}")
print(f"  Samples:  {X_sel.shape[0]}")

# TODO: Standardise X_sel before L1 fitting. L1 is scale-sensitive!
# Hint: StandardScaler().fit_transform(X_sel)
X_scaled = ____

C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0]



## TASK 3 — TRAIN: walk the regularisation path


In [ ]:
print("\n--- Regularisation Path ---")
print(f"{'C':>8} {'Non-zero':>10} / {'Total':>6}")
print("-" * 30)

lasso_results: dict[float, dict] = {}
for c_val in C_VALUES:
    # TODO: Fit a LogisticRegression with penalty="l1", solver="saga",
    # C=c_val, max_iter=5000, random_state=42.
    # Hint: build the estimator then call .fit(X_scaled, y_binary)
    lasso = ____
    ____

    coefs = lasso.coef_[0].copy()
    n_nonzero = int((np.abs(coefs) > 1e-6).sum())
    lasso_results[c_val] = {"n_nonzero": n_nonzero, "coefs": coefs}
    print(f"  {c_val:>8.3f} {n_nonzero:>10} / {len(feature_cols):>6}")


LASSO_C = 0.1
lasso_coefs = lasso_results[LASSO_C]["coefs"]

# TODO: Build the selected feature list from non-zero coefficients.
# Hint: [name for name, coef in zip(feature_cols, lasso_coefs) if abs(coef) > 1e-6]
lasso_selected = ____

lasso_importance = sorted(
    [
        (name, float(abs(coef)))
        for name, coef in zip(feature_cols, lasso_coefs)
        if abs(coef) > 1e-6
    ],
    key=lambda x: x[1],
    reverse=True,
)



### Checkpoint 1


In [ ]:
assert len(lasso_selected) > 0, "Task 3: L1 should retain SOME features"
assert len(lasso_selected) < len(
    feature_cols
), "Task 3: L1 should ELIMINATE some features"
print("\n[ok] Checkpoint 1 passed — regularisation path complete\n")



## TASK 4 — VISUALISE sparsity trajectory


In [ ]:
print(f"\n--- L1 Selected Features (C={LASSO_C}, {len(lasso_selected)} total) ---")
print(f"{'Feature':<35} {'|coef|':>10}")
print("-" * 48)
max_abs = max((c for _, c in lasso_importance), default=1.0)
for name, imp in lasso_importance[:15]:
    bar = "#" * int((imp / max_abs) * 20)
    print(f"  {name:<33} {imp:>10.4f}  {bar}")

print("\n--- Sparsity Trajectory ---")
for c_val in C_VALUES:
    nz = lasso_results[c_val]["n_nonzero"]
    bar = "#" * int(nz / max(1, len(feature_cols)) * 40)
    print(f"  C={c_val:>7.3f}  {nz:>4}/{len(feature_cols):<4}  {bar}")



### Checkpoint 2


In [ ]:
assert (
    lasso_results[0.001]["n_nonzero"] <= lasso_results[10.0]["n_nonzero"]
), "Task 4: stronger regularisation must yield fewer non-zero coefficients"
print("\n[ok] Checkpoint 2 passed — sparsity trajectory is monotonic\n")



## TASK 4b — LOG the embedded run


In [ ]:
async def log_embedded() -> str:
    conn, tracker, exp_id = await setup_tracking()
    run_id = await log_selection_run(
        tracker,
        exp_id,
        run_name=f"embedded_lasso_c{LASSO_C}",
        method="embedded",
        selected_features=lasso_selected,
        total_features=len(feature_cols),
        extra_params={
            "estimator": "LogisticRegression",
            "penalty": "l1",
            "C": str(LASSO_C),
            "solver": "saga",
            "max_iter": "5000",
        },
        extra_metrics={
            "top_abs_coef": lasso_importance[0][1] if lasso_importance else 0.0,
            "n_nonzero_c0p001": float(lasso_results[0.001]["n_nonzero"]),
            "n_nonzero_c10": float(lasso_results[10.0]["n_nonzero"]),
        },
    )
    await conn.close()
    return run_id


run_id  = await log_embedded()
print(f"\n  ExperimentTracker run: {run_id}")



## TASK 5 — APPLY: DBS Bank Card-Fraud Scoring


In [ ]:
# DBS processes ~6M card transactions/day. L1 wins because:
#   - single-fit training fits a nightly retrain budget
#   - sparse weights skip zeroed features at inference time
#   - linear coefficients are directly auditable by MAS regulators
#
# BUSINESS IMPACT: ~S$3.36M/year in additional fraud prevented; ~11x
# net ROI after infra ops.



## REFLECTION


[x] Standardised features before L1 fitting (non-negotiable)
  [x] Walked the regularisation path across multiple C values
  [x] Read non-zero coefficients as an interpretable ranking

  Next: 05_validation_and_tracking.py — schema contract, multi-method
  consensus, and leakage audit.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

